Cell 1: Installation & Setup

In [1]:
# Install necessary libraries
!pip install transformers datasets accelerate scipy scikit-learn sentencepiece

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    set_seed
)
from scipy.stats import pearsonr
from google.colab import files
import io

# Set deterministic seed for reproducibility
set_seed(42)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ System Ready. Using Device: {device}")

✅ System Ready. Using Device: cuda


Cell 2: The Core System (Model + Hybrid Loss)

In [3]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    MODEL_NAME = "microsoft/deberta-v3-base"
    MAX_LEN = 128
    BATCH_SIZE = 8       # Standard for T4 GPU
    LEARNING_RATE = 2e-5
    EPOCHS = 3
    WEIGHT_DECAY = 0.01

# ==========================================
# 2. HYBRID LOSS FUNCTION (MSE + CCC)
# ==========================================
class HybridLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()

    def ccc_loss(self, y_true, y_pred):
        # Pearson Correlation components
        x_mean = torch.mean(y_true)
        y_mean = torch.mean(y_pred)
        covariance = torch.mean((y_true - x_mean) * (y_pred - y_mean))
        x_var = torch.var(y_true)
        y_var = torch.var(y_pred)
        x_std = torch.std(y_true)
        y_std = torch.std(y_pred)

        # Calculate CCC
        rho = covariance / (x_std * y_std + 1e-8)
        numerator = 2 * rho * x_std * y_std
        denominator = x_var + y_var + (x_mean - y_mean)**2
        return 1.0 - (numerator / (denominator + 1e-8))

    def forward(self, inputs, targets):
        loss_mse = self.mse(inputs, targets)
        # Average CCC for Valence (idx 0) and Arousal (idx 1)
        loss_ccc_v = self.ccc_loss(targets[:, 0], inputs[:, 0])
        loss_ccc_a = self.ccc_loss(targets[:, 1], inputs[:, 1])
        loss_ccc = (loss_ccc_v + loss_ccc_a) / 2

        return self.alpha * loss_mse + (1 - self.alpha) * loss_ccc

# ==========================================
# 3. DATASET HELPER
# ==========================================
class DABSADataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.text = df['text'].values
        self.aspect = df['aspect'].values
        if not is_test:
            self.targets = df[['valence', 'arousal']].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text = str(self.text[index])
        aspect = str(self.aspect[index])
        inputs = self.tokenizer(
            text, aspect,
            truncation=True, padding='max_length', max_length=self.max_len,
            return_tensors="pt"
        )
        item = {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
        }
        if not self.is_test:
            item['labels'] = torch.tensor(self.targets[index], dtype=torch.float)
        return item

# ==========================================
# 4. MODEL DEFINITION
# ==========================================
class DebertaForDABSA(nn.Module):
    def __init__(self):
        super(DebertaForDABSA, self).__init__()
        self.deberta = AutoModel.from_pretrained(Config.MODEL_NAME)
        self.drop = nn.Dropout(0.1)
        self.fc = nn.Linear(768, 2) # 768 is DeBERTa base hidden size
        self.loss_fn = HybridLoss(alpha=0.5)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :] # CLS token
        logits = self.fc(self.drop(pooled_output))

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return (loss, logits) if loss is not None else logits

# ==========================================
# 5. CUSTOM TRAINER
# ==========================================
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss, logits = outputs
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p):
    preds = p.predictions[1] if isinstance(p.predictions, tuple) else p.predictions
    labels = p.label_ids
    rmse = np.sqrt(((preds - labels) ** 2).mean())
    pcc_v, _ = pearsonr(labels[:, 0], preds[:, 0])
    pcc_a, _ = pearsonr(labels[:, 1], preds[:, 1])
    return {"rmse": rmse, "pcc_valence": pcc_v, "pcc_arousal": pcc_a}

Cell 3: Upload Data & Run Training

In [10]:
import os
import json
import pandas as pd
import torch
import torch.nn as nn
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModel, set_seed
from google.colab import files
from scipy.stats import pearsonr
import numpy as np

# 1. SETUP & CONFIGURATION
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using Device: {device}")

class Config:
    MODEL_NAME = "microsoft/deberta-v3-base"
    MAX_LEN = 128
    # Using Batch Size 4 + Grad Accumulation 2 = Effective Batch Size 8
    # This prevents OOM errors while keeping math identical to the paper
    BATCH_SIZE = 4
    GRAD_ACCUMULATION = 2
    LEARNING_RATE = 2e-5
    EPOCHS = 3
    WEIGHT_DECAY = 0.01

# 2. CHECK & LOAD DATA
expected_files = ['eng_laptop_train_alltasks (1).jsonl', 'eng_restaurant_train_alltasks (1).jsonl']

# Simple check to see if files exist, if not, prompt upload
missing = [f for f in expected_files if f not in os.listdir('.')]
if missing:
    print("👉 Please upload your JSONL files now...")
    uploaded = files.upload()

data_rows = []
print("Processing data...")
for filename in expected_files:
    if filename not in os.listdir('.'): continue
    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()
    for line in content.strip().split('\n'):
        try:
            obj = json.loads(line)
            text = obj['Text']
            for q in obj.get('Quadruplet', []):
                aspect = q['Aspect']
                if aspect == 'NULL': continue
                valence, arousal = q['VA'].split('#')
                data_rows.append({
                    'text': text, 'aspect': aspect,
                    'valence': float(valence), 'arousal': float(arousal)
                })
        except: continue

df = pd.DataFrame(data_rows)
print(f"✅ Dataset Loaded: {len(df)} samples")

# 3. DATASET & MODEL DEFINITION
class DABSADataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.text = df['text'].values
        self.aspect = df['aspect'].values
        self.targets = df[['valence', 'arousal']].values

    def __len__(self): return len(self.df)

    def __getitem__(self, index):
        inputs = self.tokenizer(
            str(self.text[index]), str(self.aspect[index]),
            truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.targets[index], dtype=torch.float)
        }

class DebertaForDABSA(nn.Module):
    def __init__(self):
        super(DebertaForDABSA, self).__init__()
        self.deberta = AutoModel.from_pretrained(Config.MODEL_NAME)
        self.drop = nn.Dropout(0.1)
        self.fc = nn.Linear(768, 2)
        self.mse = nn.MSELoss()

    # CCC Loss Implementation
    def ccc_loss(self, y_true, y_pred):
        x_mean, y_mean = torch.mean(y_true), torch.mean(y_pred)
        covariance = torch.mean((y_true - x_mean) * (y_pred - y_mean))
        x_var, y_var = torch.var(y_true), torch.var(y_pred)
        x_std, y_std = torch.std(y_true), torch.std(y_pred)
        rho = covariance / (x_std * y_std + 1e-8)
        numerator = 2 * rho * x_std * y_std
        denominator = x_var + y_var + (x_mean - y_mean)**2
        return 1.0 - (numerator / (denominator + 1e-8))

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        # Use CLS token
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.fc(self.drop(cls_output))

        loss = None
        if labels is not None:
            # Hybrid Loss: 0.5 * MSE + 0.5 * CCC
            mse = self.mse(logits, labels)
            ccc = (self.ccc_loss(labels[:,0], logits[:,0]) + self.ccc_loss(labels[:,1], logits[:,1])) / 2
            loss = 0.5 * mse + 0.5 * ccc

        return (loss, logits) if loss is not None else logits

# 4. TRAINING SETUP
train_df = df.sample(frac=0.9, random_state=42)
val_df = df.drop(train_df.index)

tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
train_dataset = DABSADataset(train_df, tokenizer, Config.MAX_LEN)
val_dataset = DABSADataset(val_df, tokenizer, Config.MAX_LEN)

model = DebertaForDABSA()
model.to(device) # Ensure model is on GPU
model.float()    # Force model to Float32 to prevent "Half" errors

def compute_metrics(p):
    preds, labels = p.predictions[1] if isinstance(p.predictions, tuple) else p.predictions, p.label_ids
    rmse = np.sqrt(((preds - labels)**2).mean())
    pcc_v, _ = pearsonr(labels[:, 0], preds[:, 0])
    pcc_a, _ = pearsonr(labels[:, 1], preds[:, 1])
    return {"rmse": rmse, "pcc_valence": pcc_v, "pcc_arousal": pcc_a}

# Custom Trainer to handle tuple outputs
class FixedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss, logits = outputs
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=Config.EPOCHS,
    per_device_train_batch_size=Config.BATCH_SIZE,
    gradient_accumulation_steps=Config.GRAD_ACCUMULATION,
    learning_rate=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY,

    # --- CRITICAL STABILITY SETTINGS ---
    fp16=False,             # FORCE OFF to prevent "Half and Float" error
    eval_strategy="epoch",  # Correct new syntax
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = FixedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("\n🚀 Starting Training...")
trainer.train()

# Save Final Model
torch.save(model.state_dict(), "hausanlp_model.bin")
print("\n✅ Training Complete! Model saved as 'hausanlp_model.bin'")

✅ Using Device: cuda
Processing data...
✅ Dataset Loaded: 7298 samples


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 Starting Training...


Epoch,Training Loss,Validation Loss,Rmse,Pcc Valence,Pcc Arousal
1,1.424598,1.787886,1.604921,-0.041602,0.027158
2,1.418460,1.664251,1.525907,-0.063644,-0.019310
3,1.725100,1.663168,1.525233,0.034655,0.085544


/tmp/ipython-input-1694266991.py:131: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  pcc_v, _ = pearsonr(labels[:, 0], preds[:, 0])
/tmp/ipython-input-1694266991.py:132: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  pcc_a, _ = pearsonr(labels[:, 1], preds[:, 1])
/tmp/ipython-input-1694266991.py:131: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  pcc_v, _ = pearsonr(labels[:, 0], preds[:, 0])
/tmp/ipython-input-1694266991.py:132: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  pcc_a, _ = pearsonr(labels[:, 1], preds[:, 1])



✅ Training Complete! Model saved as 'hausanlp_model.bin'


In [16]:
import os
import json
import torch
import shutil
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader

# --- CONFIGURATION ---
# Using the filenames you uploaded
TEST_FILES = {
    'laptop': 'eng_laptop_test_task1.jsonl',
    'restaurant': 'eng_restaurant_test_task1.jsonl'
}
MODEL_PATH = "hausanlp_model.bin"
OUTPUT_DIR = "subtask_1"
ZIP_NAME = "submission"

# --- CHECK FILES ---
print("Checking for files...")
missing = [f for f in TEST_FILES.values() if f not in os.listdir('.')]
if missing:
    print(f"❌ STOP! Missing files: {missing}")
    print("Please upload 'eng_laptop_test_task1.jsonl' and 'eng_restaurant_test_task1.jsonl'")
else:
    print("✅ Test files found.")
    if not os.path.exists(MODEL_PATH):
        print(f"❌ STOP! Model file '{MODEL_PATH}' not found. Did you run the training cell?")
    else:
        print("✅ Model file found. Starting prediction...")

        # --- MODEL CLASS ---
        class DebertaForDABSA(torch.nn.Module):
            def __init__(self):
                super(DebertaForDABSA, self).__init__()
                self.deberta = AutoModel.from_pretrained("microsoft/deberta-v3-base")
                self.drop = torch.nn.Dropout(0.1)
                self.fc = torch.nn.Linear(768, 2)

            def forward(self, input_ids, attention_mask):
                outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
                return self.fc(self.drop(outputs.last_hidden_state[:, 0, :]))

        # --- INFERENCE DATASET ---
        class InferenceDataset(Dataset):
            def __init__(self, texts, aspects, tokenizer, max_len=128):
                self.texts = texts
                self.aspects = aspects
                self.tokenizer = tokenizer
                self.max_len = max_len

            def __len__(self): return len(self.texts)

            def __getitem__(self, item):
                inputs = self.tokenizer(
                    str(self.texts[item]), str(self.aspects[item]),
                    truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt"
                )
                return {
                    'input_ids': inputs['input_ids'].flatten(),
                    'attention_mask': inputs['attention_mask'].flatten()
                }

        # --- GENERATION LOGIC ---
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")

        tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
        model = DebertaForDABSA()

        # Load weights
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

        # --- THE FIX IS HERE ---
        model.to(device)
        model.float()   # <--- Forces everything to Float32 to prevent mismatch errors
        model.eval()
        # -----------------------

        if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
        os.makedirs(OUTPUT_DIR)

        for domain, filename in TEST_FILES.items():
            print(f"\nProcessing {domain}...")
            with open(filename, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            ids, texts, aspects = [], [], []

            for line in lines:
                obj = json.loads(line)

                # Check for "Aspect" list (Standard for Task 1 Test Data)
                aspect_list = []
                if 'Aspect' in obj and isinstance(obj['Aspect'], list):
                    aspect_list = obj['Aspect']
                elif 'Quadruplet' in obj:
                    aspect_list = [q['Aspect'] for q in obj['Quadruplet']]

                for asp in aspect_list:
                    if asp == 'NULL': continue
                    ids.append(obj['ID'])
                    texts.append(obj['Text'])
                    aspects.append(asp)

            dataset = InferenceDataset(texts, aspects, tokenizer)
            loader = DataLoader(dataset, batch_size=16, shuffle=False)

            all_preds = []
            with torch.no_grad():
                for batch in loader:
                    input_ids = batch['input_ids'].to(device)
                    mask = batch['attention_mask'].to(device)
                    outputs = model(input_ids, mask)
                    all_preds.extend(outputs.cpu().numpy())

            # Format Results
            results_map = {}
            for i, (p_val, p_aro) in enumerate(all_preds):
                rid = ids[i]
                # Enforce valid range [1.0, 9.0]
                val = max(1.0, min(9.0, p_val))
                aro = max(1.0, min(9.0, p_aro))

                if rid not in results_map: results_map[rid] = []
                results_map[rid].append({
                    "Aspect": aspects[i],
                    "VA": f"{val:.2f}#{aro:.2f}"
                })

            # Save to pred_eng_{domain}.jsonl
            output_filename = f"{OUTPUT_DIR}/pred_eng_{domain}.jsonl"
            with open(output_filename, 'w', encoding='utf-8') as f:
                for rid, predictions in results_map.items():
                    f.write(json.dumps({"ID": rid, "Aspect_VA": predictions}) + "\n")

            print(f"✅ Generated {output_filename}")

        # Zip it up
        shutil.make_archive(ZIP_NAME, 'zip', OUTPUT_DIR)
        print(f"\n🎉 SUCCESS! Download '{ZIP_NAME}.zip' and upload it to Codabench.")

Checking for files...
✅ Test files found.
✅ Model file found. Starting prediction...
Using device: cuda


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Processing laptop...
✅ Generated subtask_1/pred_eng_laptop.jsonl

Processing restaurant...
✅ Generated subtask_1/pred_eng_restaurant.jsonl

🎉 SUCCESS! Download 'submission.zip' and upload it to Codabench.
